## 1. Instalación de Dependencias

Instalamos las librerías necesarias para el fine-tuning con LoRA:

In [ ]:
# Instalación de librerías necesarias (versiones actualizadas y compatibles)
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U accelerate
!pip install -q -U bitsandbytes
!pip install -q -U datasets
!pip install -q -U trl

print("Todas las dependencias instaladas correctamente")

### Subir el archivo de dataset

**IMPORTANTE**: Sube el archivo `tutor_dataset.jsonl` a Colab usando el panel de archivos (icono de carpeta a la izquierda)

O ejecuta esta celda para subirlo:

In [ ]:
from google.colab import files

print("Selecciona el archivo tutor_dataset.jsonl desde tu computadora")
uploaded = files.upload()
print("Archivo subido correctamente")

## 2. Carga y Preparación del Dataset

Cargamos el dataset y lo formateamos para Phi-3:

In [ ]:
import json
from datasets import Dataset
import pandas as pd

# Cargar el dataset JSONL
data = []
with open('tutor_dataset.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Dataset cargado: {len(data)} ejemplos")
print(f"\nEjemplo del primer dato:")
print(f"Prompt: {data[0]['prompt'][:100]}...")
print(f"Response: {data[0]['response'][:100]}...")

In [ ]:
# Función para formatear datos en el formato de instrucciones de Phi-3
def format_instruction(sample):
    """Formatea el prompt y la respuesta en el formato de chat de Phi-3"""
    instruction_text = f"""<|system|>
Eres un tutor experto en Python especializado en enseñar programación a estudiantes de primer semestre de ingeniería. 
Tu objetivo es explicar conceptos de manera clara, con ejemplos prácticos y detectando errores comunes.
Siempre incluye:
- Explicación conceptual
- Ejemplos de código
- Casos de uso prácticos
- Errores comunes a evitar<|end|>
<|user|>
{sample['prompt']}<|end|>
<|assistant|>
{sample['response']}<|end|>"""
    return instruction_text

# Formatear todos los datos
formatted_data = []
for sample in data:
    formatted_data.append({
        'text': format_instruction(sample)
    })

# Convertir a Dataset de Hugging Face
dataset = Dataset.from_pandas(pd.DataFrame(formatted_data))

# Dividir en entrenamiento y validación (90-10)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Dataset formateado:")
print(f"   - Entrenamiento: {len(dataset['train'])} ejemplos")
print(f"   - Validación: {len(dataset['test'])} ejemplos")
print(f"\nEjemplo formateado:")
print(dataset['train'][0]['text'][:500] + "...")

## 3. Configuración del Modelo Phi-3 con LoRA

Configuramos Phi-3 con cuantización de 4 bits y LoRA para entrenamiento eficiente:

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Verificar GPU disponible
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"   Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Nombre del modelo base
# Usamos Phi-3-mini que es completamente abierto, no requiere autenticación
# y tiene excelente rendimiento en Colab
model_name = "microsoft/Phi-3-mini-4k-instruct"

# Configuración de cuantización a 4 bits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Cargando modelo Phi-3 (esto puede tardar unos minutos)...")

# Cargar modelo con cuantización
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Modelo cargado correctamente")

In [ ]:
# Preparar modelo para entrenamiento con LoRA
model = prepare_model_for_kbit_training(model)

# Configuración de LoRA
lora_config = LoraConfig(
    r=16,  # Rank de las matrices LoRA (aumentar para más capacidad, pero más memoria)
    lora_alpha=32,  # Escala de LoRA
    target_modules=[  # Módulos a los que aplicar LoRA
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Aplicar LoRA al modelo
model = get_peft_model(model, lora_config)

# Mostrar parámetros entrenables
trainable_params = 0
all_params = 0
for _, param in model.named_parameters():
    all_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"\nEstadísticas del modelo:")
print(f"   Parámetros totales: {all_params:,}")
print(f"   Parámetros entrenables: {trainable_params:,}")
print(f"   Porcentaje entrenable: {100 * trainable_params / all_params:.2f}%")
print(f"\nLoRA configurado correctamente")

## 4. Entrenamiento (Fine-Tuning)

Configuramos y ejecutamos el entrenamiento:

In [ ]:
# Configuración de entrenamiento optimizada para Colab
training_args = TrainingArguments(
    output_dir="./tutor_python_llama3_lora",
    num_train_epochs=3,  # Número de épocas
    per_device_train_batch_size=2,  # Tamaño de batch (ajustar según GPU)
    gradient_accumulation_steps=4,  # Acumular gradientes para simular batch más grande
    gradient_checkpointing=True,  # Ahorra memoria
    optim="paged_adamw_8bit",  # Optimizador eficiente en memoria
    save_steps=25,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,  # Usar bfloat16 si está disponible
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none",  # Cambiar a "tensorboard" si quieres usar TensorBoard
    eval_strategy="steps",
    eval_steps=25,
    save_total_limit=2,  # Solo guardar los últimos 2 checkpoints
)

print("Configuración de entrenamiento lista")

In [ ]:
# Inicializar el Trainer con parámetros mínimos
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=training_args,
)

print("Trainer inicializado")
print("\nIniciando entrenamiento...")
print("Esto puede tardar varios minutos dependiendo de la GPU\n")

In [ ]:
# ENTRENAR EL MODELO
trainer.train()

print("\n\nEntrenamiento completado!")

## 5. Guardar el Modelo Entrenado

Guardamos el modelo y los adaptadores LoRA:

In [ ]:
# Guardar el modelo entrenado
output_dir = "./tutor_python_final"

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Modelo guardado en: {output_dir}")
print(f"\nArchivos guardados:")
!ls -lh {output_dir}

### Descargar el modelo a tu computadora

Comprime y descarga el modelo entrenado:

In [ ]:
# Comprimir el modelo
!zip -r tutor_python_final.zip {output_dir}

# Descargar
from google.colab import files
files.download('tutor_python_final.zip')

print("Modelo comprimido y listo para descargar")

## 6. Evaluación y Pruebas del Tutor

Probamos el tutor con preguntas de ejemplo:

In [ ]:
# Función para generar respuestas del tutor
def preguntar_tutor(pregunta, max_length=512, temperature=0.7):
    """
    Genera una respuesta del tutor para la pregunta dada.
    
    Args:
        pregunta: La pregunta del estudiante
        max_length: Longitud máxima de la respuesta
        temperature: Controla la creatividad (0.0 = determinista, 1.0 = creativo)
    """
    # Formatear la pregunta
    prompt = f"""<|system|>
Eres un tutor experto en Python especializado en enseñar programación a estudiantes de primer semestre de ingeniería. 
Tu objetivo es explicar conceptos de manera clara, con ejemplos prácticos y detectando errores comunes.
Siempre incluye:
- Explicación conceptual
- Ejemplos de código
- Casos de uso prácticos
- Errores comunes a evitar<|end|>
<|user|>
{pregunta}<|end|>
<|assistant|>
"""
    
    # Tokenizar
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generar respuesta
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False  # Evita problemas de compatibilidad con DynamicCache
        )
    
    # Decodificar y extraer solo la respuesta del asistente
    respuesta_completa = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extraer solo la parte del asistente
    if "<|assistant|>" in respuesta_completa:
        respuesta = respuesta_completa.split("<|assistant|>")[1]
        respuesta = respuesta.split("<|end|>")[0].strip()
    else:
        respuesta = respuesta_completa
    
    return respuesta

print("Función de inferencia lista")

In [ ]:
# Preguntas de prueba
preguntas_test = [
    "¿Qué es una variable en Python?",
    "¿Cómo funciona un ciclo for?",
    "¿Cuál es la diferencia entre = y == en Python?",
    "¿Qué es una lista y cómo la uso?",
    "Mi código tiene un error de indentación, ¿qué significa eso?"
]

print("EVALUACIÓN DEL TUTOR\n")
print("="*80)

for i, pregunta in enumerate(preguntas_test, 1):
    print(f"\nPregunta {i}: {pregunta}")
    print("-"*80)
    
    respuesta = preguntar_tutor(pregunta, max_length=400)
    print(f"Respuesta del tutor:\n")
    print(respuesta)
    print("\n" + "="*80)

## 7. Interfaz Interactiva

Prueba el tutor de forma interactiva:

In [ ]:
print("TUTOR INTERACTIVO DE PYTHON")
print("="*80)
print("Escribe tus preguntas sobre Python (escribe 'salir' para terminar)\n")

while True:
    pregunta = input("\nTu pregunta: ")
    
    if pregunta.lower() in ['salir', 'exit', 'quit']:
        print("\nHasta luego! Sigue practicando Python.")
        break
    
    if not pregunta.strip():
        continue
    
    print("\nTutor: Procesando...")
    respuesta = preguntar_tutor(pregunta, max_length=400)
    print(f"\n{respuesta}")
    print("\n" + "-"*80)

## 8. Métricas de Evaluación (Opcional)

Evaluamos el modelo en el conjunto de validación:

In [ ]:
# Evaluar el modelo
eval_results = trainer.evaluate()

print("RESULTADOS DE EVALUACIÓN")
print("="*80)
for key, value in eval_results.items():
    print(f"   {key}: {value:.4f}")

# Calcular perplexity
import math
perplexity = math.exp(eval_results['eval_loss'])
print(f"\n   Perplexity: {perplexity:.4f}")
print("\nMenor perplexity = mejor modelo")

## 9. Comparación Antes vs Después

Comparamos el modelo base sin entrenar vs el modelo fine-tuneado:

In [ ]:
# Cargar modelo base sin LoRA para comparación
print("Cargando modelo base para comparación...")

model_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

def preguntar_modelo_base(pregunta, max_length=512):
    """Genera respuesta del modelo base sin fine-tuning"""
    prompt = f"""<|system|>
Eres un tutor experto en Python.<|end|>
<|user|>
{pregunta}<|end|>
<|assistant|>
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model_base.device)
    
    with torch.no_grad():
        outputs = model_base.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_length,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,  # Evita problemas de compatibilidad
            past_key_values=None  # Fuerza no usar caché
        )
    
    respuesta_completa = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    if "<|assistant|>" in respuesta_completa:
        respuesta = respuesta_completa.split("<|assistant|>")[1]
        respuesta = respuesta.split("<|end|>")[0].strip()
    else:
        respuesta = respuesta_completa
    
    return respuesta

print("Modelo base cargado")

In [ ]:
# Comparar respuestas
pregunta_comparacion = "¿Qué es una variable en Python?"

print("COMPARACIÓN: MODELO BASE vs MODELO FINE-TUNEADO")
print("="*80)
print(f"\nPregunta: {pregunta_comparacion}\n")

print("-"*80)
print("MODELO BASE (sin entrenamiento):")
print("-"*80)
respuesta_base = preguntar_modelo_base(pregunta_comparacion, max_length=400)
print(respuesta_base)

print("\n" + "="*80)
print("MODELO FINE-TUNEADO (con tu dataset):")
print("-"*80)
respuesta_tuneada = preguntar_tutor(pregunta_comparacion, max_length=400)
print(respuesta_tuneada)

print("\n" + "="*80)

## 10. Análisis del Dataset

Estadísticas del dataset utilizado:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Analizar longitud de prompts y responses
prompt_lengths = [len(item['prompt']) for item in data]
response_lengths = [len(item['response']) for item in data]

# Crear visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de prompts
axes[0].hist(prompt_lengths, bins=20, color='skyblue', edgecolor='black')
axes[0].set_title('Distribución de Longitud de Preguntas')
axes[0].set_xlabel('Caracteres')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(np.mean(prompt_lengths), color='red', linestyle='--', label=f'Media: {np.mean(prompt_lengths):.0f}')
axes[0].legend()

# Histograma de responses
axes[1].hist(response_lengths, bins=20, color='lightgreen', edgecolor='black')
axes[1].set_title('Distribución de Longitud de Respuestas')
axes[1].set_xlabel('Caracteres')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(np.mean(response_lengths), color='red', linestyle='--', label=f'Media: {np.mean(response_lengths):.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nESTADÍSTICAS DEL DATASET:")
print(f"   Total de ejemplos: {len(data)}")
print(f"   Prompts - Media: {np.mean(prompt_lengths):.0f} caracteres")
print(f"   Prompts - Min/Max: {min(prompt_lengths)}/{max(prompt_lengths)} caracteres")
print(f"   Responses - Media: {np.mean(response_lengths):.0f} caracteres")
print(f"   Responses - Min/Max: {min(response_lengths)}/{max(response_lengths)} caracteres")

## 11. Inspección de Tokens del Modelo

Verificamos cómo el modelo tokeniza las entradas:

In [ ]:
# Ejemplo de tokenización
texto_ejemplo = "¿Qué es una variable en Python?"

tokens = tokenizer.tokenize(texto_ejemplo)
ids = tokenizer.encode(texto_ejemplo)

print("ANÁLISIS DE TOKENIZACIÓN:")
print("="*80)
print(f"Texto original: {texto_ejemplo}")
print(f"\nNúmero de tokens: {len(tokens)}")
print(f"\nTokens: {tokens}")
print(f"\nIDs de tokens: {ids}")

# Estadísticas de tokenización del dataset
token_counts = []
for sample in data:
    texto_completo = sample['prompt'] + " " + sample['response']
    tokens_count = len(tokenizer.encode(texto_completo))
    token_counts.append(tokens_count)

print(f"\nTOKENS EN EL DATASET:")
print(f"   Media: {np.mean(token_counts):.0f} tokens")
print(f"   Máximo: {max(token_counts)} tokens")
print(f"   Mínimo: {min(token_counts)} tokens")

## 12. Consejos para Mejorar el Tutor

### Para mejorar aún más el modelo:

1. **Expandir el dataset**:
   - Agregar más ejemplos (objetivo: 200-500)
   - Incluir errores comunes y sus soluciones
   - Agregar ejercicios prácticos con soluciones paso a paso

2. **Ajustar hiperparámetros**:
   - Aumentar `num_train_epochs` a 5-10
   - Experimentar con diferentes valores de `learning_rate`
   - Ajustar `r` y `lora_alpha` en LoRA

3. **Usar un modelo más grande**:
   - `Llama-3.2-3B-Instruct` (requiere más memoria)
   - Considerar Colab Pro para GPUs más potentes

## 13. Cargar el Modelo Guardado (Para Uso Futuro)

Código para cargar el modelo entrenado en una sesión futura:

In [ ]:
# Este código muestra cómo cargar el modelo en el futuro

from peft import PeftModel

# Cargar modelo base
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Cargar adaptadores LoRA
model_cargado = PeftModel.from_pretrained(base_model, "./tutor_python_final")

# Cargar tokenizer
tokenizer_cargado = AutoTokenizer.from_pretrained("./tutor_python_final")

print("Modelo cargado desde disco")

# Ahora puedes usar model_cargado igual que model